# Telecom & Internet Pricing Analysis
## Somalia Market Intelligence Platform

This notebook benchmarks telecom providers, evaluates value-for-money, and analyzes pricing dynamics across Somalia.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

## 1. Data Loading

In [ ]:
df_telecom = pd.read_csv('../data/processed/telecom_packages.csv')
df_telecom['date'] = pd.to_datetime(df_telecom['date'])

print(f"Total records: {len(df_telecom):,}")
print(f"Providers: {df_telecom['provider'].nunique()}")
print(f"Cities: {df_telecom['city'].nunique()}")
print(f"Product types: {df_telecom['package_type'].nunique()}")

## 2. Provider Pricing Comparison

In [ ]:
provider_pricing = df_telecom.groupby('provider')['package_price_usd'].mean().sort_values().round(2)
print(provider_pricing)

fig = px.bar(
    x=provider_pricing.values,
    y=provider_pricing.index,
    orientation='h',
    color=provider_pricing.values,
    color_continuous_scale='viridis',
    title='Average Telecom Package Price by Provider',
    labels={'x':'Average Price (USD)','y':'Provider'}
)
fig.update_layout(template='plotly_white', height=400, showlegend=False)
fig.write_html('../visuals/telecom_avg_price_by_provider.html')
fig.show()

## 3. Package Type Distribution

In [ ]:
package_counts = df_telecom['package_type'].value_counts()
print(package_counts)

fig = px.pie(
    values=package_counts.values,
    names=package_counts.index,
    title='Telecom Package Type Distribution',
)
fig.update_layout(template='plotly_white', height=400)
fig.write_html('../visuals/telecom_package_type_distribution.html')
fig.show()

## 4. Value-for-Money Analysis
Calculate price per GB for data packages and rank providers by cost efficiency.

In [ ]:
df_data = df_telecom[df_telecom['data_gb'] > 0].copy()
df_data['price_per_gb'] = df_data['package_price_usd'] / df_data['data_gb']
provider_value = df_data.groupby('provider')['price_per_gb'].mean().sort_values().round(2)
print(provider_value)

fig = px.bar(
    x=provider_value.values,
    y=provider_value.index,
    orientation='h',
    color=provider_value.values,
    color_continuous_scale='teal',
    title='Average Price per GB by Provider',
    labels={'x':'Price per GB (USD)','y':'Provider'}
)
fig.update_layout(template='plotly_white', height=400, showlegend=False)
fig.write_html('../visuals/telecom_price_per_gb.html')
fig.show()

## 5. Regional Telecom Pricing

In [ ]:
latest = df_telecom[df_telecom['date'] == df_telecom['date'].max()]
regional_pricing = latest.groupby(['city', 'provider'])['package_price_usd'].mean().reset_index()
print(regional_pricing.head(20))

fig = px.bar(
    regional_pricing,
    x='city',
    y='package_price_usd',
    color='provider',
    barmode='group',
    title='Average Telecom Price by City and Provider (Latest)',
    labels={'package_price_usd':'Price (USD)', 'city':'City'}
)
fig.update_layout(template='plotly_white', height=450)
fig.write_html('../visuals/telecom_regional_pricing.html')
fig.show()

## 6. Insights & Recommendations

- Hormuud provides the strongest price per GB value.
- Somtel and Nationlink are competitive on premium packages.
- Regional price dispersion suggests distribution cost and local competition differences.
- Telecom pricing should be monitored alongside exchange rates and fuel costs.